In [1]:
#!/usr/bin/env python3
"""
npy_scalogram_classify_balanced_split_4ch.py

Train/evaluate directly from **.npy** scalograms with TRUE 4-channel fusion for vibration:
- Acoustic: single-channel -> padded to 4 channels (signal in ch0)
- Vibration: groups (x_A, y_A, x_B, y_B) per (source_csv, window_index) stacked as (C<=4, F, T)

Steps:
- Balanced sampling per modality & condition (cap TARGET_PER_CONDITION)
- Stratified 70/15/15 split by condition
- Normalization:
    * CNN: log1p + per-channel z-score
    * BPNN: log1p + flatten channels (pad to 4) -> PCA -> MLP baseline
- Stabilized CNN training (AdamW, cosine sched, label smoothing, dropout, BN, AMP, clip, early stop)

Run:
  python npy_scalogram_classify_balanced_split_4ch.py
"""

import os, glob, random, json
from pathlib import Path
from typing import List, Tuple, Dict, Optional

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# -------------------- CONFIG --------------------
SCALOGRAM_ROOT = "scalograms"
ACOUSTIC_DIR   = os.path.join(SCALOGRAM_ROOT, "acoustic")
VIBRATION_DIR  = os.path.join(SCALOGRAM_ROOT, "vibration")

# Channel order for vibration fusion
VIB_CH_ORDER = [
    "x_direction_housing_A",
    "y_direction_housing_A",
    "x_direction_housing_B",
    "y_direction_housing_B",
]

TARGET_PER_CONDITION = 24          # cap per modality per condition (take all if fewer)

# Thinning / cropping
FREQ_SLICE = (None, 128)           # (start, end) rows; None keeps edge
MAX_TIME_COLS_PER_FILE = 20000     # cap columns
COL_STRIDE = 8                     # keep every Nth time column

# Classical baseline
PCA_DIM = 64

# Torch training
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS_CNN  = 20
BATCH_SIZE  = 16
BASE_LR     = 3e-4
WEIGHT_DECAY= 1e-2
LABEL_SMOOTH= 0.1
DROPOUT_P   = 0.2
MIXED_PREC  = True
CLIP_NORM   = 1.0

# Data aug (set >0 to enable)
TIME_MASK_PCT = 0.0   # e.g., 0.05 masks 5% of time steps
FREQ_MASK_PCT = 0.0   # e.g., 0.10 masks 10% of freq bins

# Repro
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)

# -------------------- HELPERS --------------------

def parse_from_meta(meta_path: str):
    with open(meta_path, "r") as fh:
        m = json.load(fh)
    src = m.get("source_csv", "")
    stem = os.path.splitext(os.path.basename(src))[0]
    parts = stem.split("_")
    load = parts[0] if len(parts) > 0 else "UNK"
    cond = parts[1] if len(parts) > 1 else "UNK"
    if cond.lower() == "unbalalnce":
        cond = "Unbalance"
    if cond.lower() == "normal":
        cond = "Normal"
    sev = parts[2] if len(parts) > 2 else "UNK"
    ch  = m.get("channel", None)
    win = int(m.get("window_index", 0))
    return load, cond, sev, ch, win, src

def discover_acoustic_items(acoustic_dir: str) -> List[Tuple[str, str, str]]:
    """
    Returns list of (npy_path, condition, severity) for acoustic (single channel).
    """
    items = []
    for npy in sorted(glob.glob(os.path.join(acoustic_dir, "*.npy"))):
        meta = npy + ".meta.json"
        if not os.path.exists(meta):
            continue
        try:
            _, cond, sev, _, _, _ = parse_from_meta(meta)
            items.append((npy, cond, sev))
        except Exception:
            pass
    return items

def discover_vibration_groups(vib_dir: str):
    """
    Discover vibration .npy files and group them by (source_csv, window_index).
    Returns:
      groups: List[List[(npy_path, meta_path, ch_name)]]
      labels: Dict[group_index] -> (condition, severity)
    """
    buckets: Dict[Tuple[str,int], Dict[str, Tuple[str,str]]] = {}
    labels: Dict[Tuple[str,int], Tuple[str,str]] = {}
    for npy in sorted(glob.glob(os.path.join(vib_dir, "*.npy"))):
        meta = npy + ".meta.json"
        if not os.path.exists(meta): continue
        try:
            _, cond, sev, ch, win, src = parse_from_meta(meta)
        except Exception:
            continue
        key = (src, win)
        if ch is None:
            continue
        buckets.setdefault(key, {})[ch] = (npy, meta)
        labels[key] = (cond, sev)

    groups: List[List[Tuple[str,str,str]]] = []
    y: List[Tuple[str,str]] = []
    for key, chmap in buckets.items():
        present = [ch for ch in VIB_CH_ORDER if ch in chmap]
        if not present: 
            continue
        lst = [(chmap[ch][0], chmap[ch][1], ch) for ch in present]
        groups.append(lst)
        y.append(labels[key])
    return groups, y  # y[i] = (cond, sev) for groups[i]

def freq_time_thin(power: np.ndarray) -> np.ndarray:
    # input (F,T) float32
    F0, T0 = power.shape
    fs, fe  = FREQ_SLICE
    f_start = 0 if fs is None else max(0, int(fs))
    f_end   = F0 if fe is None else min(F0, int(fe))
    X = power[f_start:f_end, :]
    if MAX_TIME_COLS_PER_FILE is not None:
        T_cap = min(X.shape[1], MAX_TIME_COLS_PER_FILE)
        X = X[:, :T_cap]
    if COL_STRIDE and COL_STRIDE > 1:
        X = X[:, ::int(COL_STRIDE)]
    return X.astype(np.float32, copy=False)

def load_vibration_group_4ch(group: List[Tuple[str,str,str]]) -> np.ndarray:
    """
    group: list of (npy_path, meta_path, ch_name) with ch_name in VIB_CH_ORDER
    Returns (C<=4, F, T) stacked in VIB_CH_ORDER; time axis aligned/cropped to min T.
    """
    mats = []
    order_present = [ch for ch in VIB_CH_ORDER if any(ch == g[2] for g in group)]
    for ch in order_present:
        npy_path = [g[0] for g in group if g[2] == ch][0]
        arr = np.load(npy_path, mmap_mode="r")  # (F,T)
        X = freq_time_thin(arr)
        mats.append(X)
    # align time lengths
    Tm = min(M.shape[1] for M in mats)
    mats = [M[:, :Tm] for M in mats]
    return np.stack(mats, axis=0)  # (C, F, T)

def load_acoustic_as_4ch(npy_path: str) -> np.ndarray:
    """
    Load acoustic (F,T) and place in channel 0; zeros in other channels -> (4,F,T).
    """
    arr = np.load(npy_path, mmap_mode="r")
    X = freq_time_thin(arr)  # (F,T)
    C = 4
    out = np.zeros((C, X.shape[0], X.shape[1]), dtype=np.float32)
    out[0] = X
    return out

# -------------------- BALANCED SAMPLING --------------------

def balanced_select(acoustic_items, vib_groups, vib_labels, k=TARGET_PER_CONDITION):
    """
    Return selected sample list with unified structure:
       item = (source_type, payload, condition, severity)
       - source_type: 'acoustic' or 'vibration'
       - payload:
           * acoustic -> npy_path (str)
           * vibration -> group (List[(npy, meta, ch)])
    """
    from collections import defaultdict
    rng = np.random.default_rng(SEED)

    selected = []

    # Acoustic per condition
    by_cond_a = defaultdict(list)
    for npy_path, cond, sev in acoustic_items:
        by_cond_a[cond].append((npy_path, cond, sev))
    for cond, lst in by_cond_a.items():
        lst = lst.copy(); rng.shuffle(lst)
        take = lst[:k] if len(lst) >= k else lst
        for npy_path, c, s in take:
            selected.append(("acoustic", npy_path, c, s))

    # Vibration per condition (grouped)
    by_cond_v = defaultdict(list)
    for g, (cond, sev) in zip(vib_groups, vib_labels):
        by_cond_v[cond].append((g, cond, sev))
    for cond, lst in by_cond_v.items():
        lst = lst.copy(); rng.shuffle(lst)
        take = lst[:k] if len(lst) >= k else lst
        for g, c, s in take:
            selected.append(("vibration", g, c, s))

    rng.shuffle(selected)
    return selected

# -------------------- DATASET --------------------
class NpyDataset4Ch(Dataset):
    """
    Yields:
       X -> (4, F, T) float32 (acoustic padded to 4ch; vibration fused 4ch)
       y_cond, y_sev
    Normalization for CNN is done in the training loop (log1p + per-channel zscore).
    Here we only load & thin.
    """
    def __init__(self, items, cond2id: Dict[str,int], sev2id: Dict[str,int]):
        """
        items: list of tuples (source_type, payload, cond, sev)
        """
        self.items = items
        self.cond2id = cond2id
        self.sev2id  = sev2id

    def __len__(self): return len(self.items)

    def __getitem__(self, i):
        src_type, payload, cond, sev = self.items[i]
        if src_type == "acoustic":
            X = load_acoustic_as_4ch(payload)           # (4,F,T)
        else:
            X = load_vibration_group_4ch(payload)       # (C<=4,F,T)
            if X.shape[0] < 4:                          # pad to 4 channels
                C, F, T = X.shape
                out = np.zeros((4, F, T), dtype=np.float32)
                out[:C] = X
                X = out
        yc = self.cond2id[cond]
        ys = self.sev2id[sev]
        return torch.from_numpy(X), torch.tensor(yc, dtype=torch.long), torch.tensor(ys, dtype=torch.long)

# -------------------- MODELS --------------------
class CNN2D(nn.Module):
    def __init__(self, n_cond: int, n_sev: int, in_ch: int = 4):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(in_ch, 32, 5, padding=2, stride=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d((2,4)), nn.Dropout(DROPOUT_P),

            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d((2,2)), nn.Dropout(DROPOUT_P),

            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 8)),
        )
        self.flat = nn.Flatten()
        self.fc_cond = nn.Linear(128*4*8, n_cond)
        self.fc_sev  = nn.Linear(128*4*8, n_sev)

    def forward(self, x):
        z = self.backbone(x)
        z = self.flat(z)
        return self.fc_cond(z), self.fc_sev(z)

# -------------------- TRAIN/EVAL --------------------

def spec_mask(x: torch.Tensor) -> torch.Tensor:
    # x: (B,C,F,T)
    if TIME_MASK_PCT <= 0 and FREQ_MASK_PCT <= 0:
        return x
    B, C, F, T = x.shape
    if FREQ_MASK_PCT > 0:
        w = max(1, int(F * FREQ_MASK_PCT))
        for b in range(B):
            f0 = np.random.randint(0, max(1, F - w + 1))
            x[b, :, f0:f0+w, :] = 0
    if TIME_MASK_PCT > 0:
        w = max(1, int(T * TIME_MASK_PCT))
        for b in range(B):
            t0 = np.random.randint(0, max(1, T - w + 1))
            x[b, :, :, t0:t0+w] = 0
    return x

def normalize_batch_per_channel(x: torch.Tensor) -> torch.Tensor:
    # x: (B,C,F,T) -> log1p + per-channel zscore
    x = torch.log1p(torch.clamp(x, min=0))  # power ≥ 0 in scalograms
    mu = x.mean(dim=(2,3), keepdim=True)
    sd = x.std(dim=(2,3), keepdim=True) + 1e-6
    return (x - mu) / sd

def train_model(model, dl_tr, dl_va, epochs=EPOCHS_CNN):
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=max(1, epochs//2))
    crit = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE=="cuda" and MIXED_PREC))

    best = None; best_val = 1e9; patience=4; wait=0
    for ep in range(1, epochs+1):
        model.train(); tr_sum=0.0; n=0
        for xb, yc, ys in dl_tr:
            xb, yc, ys = xb.to(DEVICE), yc.to(DEVICE), ys.to(DEVICE)
            xb = normalize_batch_per_channel(xb)
            xb = spec_mask(xb)

            opt.zero_grad(set_to_none=True)
            if scaler.is_enabled():
                with torch.cuda.amp.autocast():
                    pc, ps = model(xb)
                    loss = crit(pc, yc) + crit(ps, ys)
                scaler.scale(loss).backward()
                nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
                scaler.step(opt); scaler.update()
            else:
                pc, ps = model(xb)
                loss = crit(pc, yc) + crit(ps, ys)
                loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
                opt.step()

            # Cosine schedule with fractional epoch
            n += xb.size(0)
            sched.step(ep - 1 + n / max(1, len(dl_tr.dataset)))

            tr_sum += float(loss.item()) * xb.size(0)
        tr = tr_sum / max(1, n)

        # val
        model.eval(); va_sum=0.0; nva=0
        with torch.no_grad():
            for xb, yc, ys in dl_va:
                xb, yc, ys = xb.to(DEVICE), yc.to(DEVICE), ys.to(DEVICE)
                xb = normalize_batch_per_channel(xb)
                pc, ps = model(xb)
                loss = crit(pc, yc) + crit(ps, ys)
                va_sum += float(loss.item()) * xb.size(0); nva += xb.size(0)
        va = va_sum / max(1,nva)
        print(f"[CNN] epoch {ep:02d}  train={tr:.4f}  val={va:.4f}")

        if va < best_val - 1e-6:
            best_val = va; best = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; wait=0
        else:
            wait += 1
            if wait >= patience:
                print("  early stop"); break

    if best is not None:
        model.load_state_dict(best)
    return model

def eval_model(model, dl):
    model.eval(); yc_true, yc_pred, ys_true, ys_pred = [], [], [], []
    with torch.no_grad():
        for xb, yc, ys in dl:
            xb = normalize_batch_per_channel(xb.to(DEVICE))
            pc, ps = model(xb)
            yc_pred.extend(pc.argmax(1).cpu().numpy().tolist())
            ys_pred.extend(ps.argmax(1).cpu().numpy().tolist())
            yc_true.extend(yc.numpy().tolist())
            ys_true.extend(ys.numpy().tolist())
    return np.array(yc_true), np.array(yc_pred), np.array(ys_true), np.array(ys_pred)

def report(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1m = f1_score(y_true, y_pred, average="macro")
    print(f"{name:>12s} | Acc={acc:.3f}  F1(macro)={f1m:.3f}")
    return acc, f1m

# -------------------- PIPELINE --------------------

def main():
    # Discover items
    acoustic_items = discover_acoustic_items(ACOUSTIC_DIR)
    vib_groups, vib_labels = discover_vibration_groups(VIBRATION_DIR)

    if not acoustic_items and not vib_groups:
        print("[ERROR] No .npy scalograms found.")
        return

    # Build vocab from all available
    conditions = sorted(set([c for _, c, _ in acoustic_items] + [c for (c, _) in vib_labels]))
    severities = sorted(set([s for _, _, s in acoustic_items] + [s for (_, s) in vib_labels]))
    cond2id = {c:i for i,c in enumerate(conditions)}
    sev2id  = {s:i for i,s in enumerate(severities)}

    # Balanced selection per modality & condition
    selected = balanced_select(acoustic_items, vib_groups, vib_labels, k=TARGET_PER_CONDITION)
    if not selected:
        # fallback: use all (treat acoustic as singles, vibration as groups)
        selected = [("acoustic", npy, c, s) for (npy, c, s) in acoustic_items] + \
                   [("vibration", g, c, s) for g, (c, s) in zip(vib_groups, vib_labels)]

    # Stratified split by condition
    y_cond = np.array([cond2id[c] for (_, _, c, _) in selected], dtype=np.int64)
    idx = np.arange(len(selected))

    def safe_split(indices, strat_labels, test_size):
        try:
            return train_test_split(indices, test_size=test_size, random_state=SEED, stratify=strat_labels)
        except Exception:
            return train_test_split(indices, test_size=test_size, random_state=SEED)

    tr_idx, te_idx = safe_split(idx, y_cond, test_size=0.15)
    tr_idx, va_idx = safe_split(tr_idx, y_cond[tr_idx], test_size=0.1765)  # ~70/15/15

    tr_items = [selected[i] for i in tr_idx]
    va_items = [selected[i] for i in va_idx]
    te_items = [selected[i] for i in te_idx]

    # Datasets & loaders
    ds_tr = NpyDataset4Ch(tr_items, cond2id, sev2id)
    ds_va = NpyDataset4Ch(va_items, cond2id, sev2id)
    ds_te = NpyDataset4Ch(te_items, cond2id, sev2id)
    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    dl_va = DataLoader(ds_va, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    dl_te = DataLoader(ds_te, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    # ===== Baseline: BPNN (MLP) on PCA features =====
    def to_flat(items_subset):
        mats = []
        for src_type, payload, _, _ in items_subset:
            if src_type == "acoustic":
                X = load_acoustic_as_4ch(payload)          # (4,F,T)
            else:
                X = load_vibration_group_4ch(payload)      # (C<=4,F,T)
                if X.shape[0] < 4:
                    C, F, T = X.shape
                    tmp = np.zeros((4, F, T), dtype=np.float32)
                    tmp[:C] = X
                    X = tmp
            X = np.log1p(X).astype(np.float32)             # per-sample log1p
            mats.append(X.reshape(-1))                      # flatten 4*F*T
        return np.stack(mats)

    Xtr_f = to_flat(tr_items); Xva_f = to_flat(va_items); Xte_f = to_flat(te_items)
    scaler = StandardScaler(with_mean=True, with_std=True)
    Xtr_s = scaler.fit_transform(Xtr_f); Xva_s = scaler.transform(Xva_f); Xte_s = scaler.transform(Xte_f)
    pca = PCA(n_components=min(PCA_DIM, Xtr_s.shape[1]), random_state=SEED)
    Xtr_p = pca.fit_transform(Xtr_s); Xva_p = pca.transform(Xva_s); Xte_p = pca.transform(Xte_s)

    yctr = np.array([cond2id[c] for (_,_,c,_) in tr_items], dtype=np.int64)
    ycva = np.array([cond2id[c] for (_,_,c,_) in va_items], dtype=np.int64)
    ycte = np.array([cond2id[c] for (_,_,c,_) in te_items], dtype=np.int64)

    mlp = MLPClassifier(hidden_layer_sizes=(256,128), activation="relu", max_iter=600, random_state=SEED)
    mlp.fit(Xtr_p, yctr)
    print("\n=== Baseline from .npy (BPNN on PCA) ===")
    report("BPNN(cond)", ycte, mlp.predict(Xte_p))

    # ===== CNN (4-channel) =====
    print("\n=== CNN from .npy (4-channel, multitask) ===")
    model = CNN2D(n_cond=len(conditions), n_sev=len(severities), in_ch=4)
    model = train_model(model, dl_tr, dl_va, epochs=EPOCHS_CNN)
    yc_true, yc_pred, ys_true, ys_pred = eval_model(model, dl_te)
    report("CNN(cond)", yc_true, yc_pred)
    report("CNN(sev)",  ys_true, ys_pred)

    print("\nDone.")

if __name__ == "__main__":
    main()



=== Baseline from .npy (BPNN on PCA) ===
  BPNN(cond) | Acc=0.897  F1(macro)=0.891

=== CNN from .npy (4-channel, multitask) ===


C:\Users\gsahi\AppData\Local\Temp\ipykernel_25448\1417927234.py:319: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE=="cuda" and MIXED_PREC))
C:\Users\gsahi\AppData\Local\Temp\ipykernel_25448\1417927234.py:331: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


[CNN] epoch 01  train=3.4718  val=3.3604
[CNN] epoch 02  train=3.0163  val=3.0557
[CNN] epoch 03  train=2.8655  val=2.8450
[CNN] epoch 04  train=2.6611  val=2.6287
[CNN] epoch 05  train=2.4854  val=2.4064
[CNN] epoch 06  train=2.3908  val=2.3351
[CNN] epoch 07  train=2.2703  val=2.2724
[CNN] epoch 08  train=2.2633  val=2.2577
[CNN] epoch 09  train=2.2432  val=2.2466
[CNN] epoch 10  train=2.2296  val=2.2584
[CNN] epoch 11  train=2.2200  val=2.0420
[CNN] epoch 12  train=2.0592  val=2.0157
[CNN] epoch 13  train=1.9216  val=1.8409
[CNN] epoch 14  train=1.8497  val=1.8084
[CNN] epoch 15  train=1.7686  val=1.7707
[CNN] epoch 16  train=1.7325  val=1.7506
[CNN] epoch 17  train=1.6804  val=1.7094
[CNN] epoch 18  train=1.6858  val=1.7157
[CNN] epoch 19  train=1.6460  val=1.7273
[CNN] epoch 20  train=1.6350  val=1.7233
   CNN(cond) | Acc=0.793  F1(macro)=0.697
    CNN(sev) | Acc=0.793  F1(macro)=0.629

Done.


In [2]:
#!/usr/bin/env python3
"""
npy_scalogram_classify_balanced_split_4ch_plus.py

Directly trains/evals from .npy scalograms with TRUE 4-channel fusion (vibration),
and padded-4ch acoustic. Balanced sampling, stratified split, standard normalization.

Models compared:
  • BPNN  (MLP on PCA features)                 [cond]
  • SVM   (RBF SVC on PCA features)             [cond]
  • ELM   (Random hidden layer on PCA features) [cond]
  • DBN   (BernoulliRBM stack + LogisticReg)    [cond]
  • AE+LR (Autoencoder on PCA -> LR)            [cond]
  • CNN   (4-ch 2D CNN, multitask)              [cond + sev]
  • GRU   (4-ch → sequence across time, multitask) [cond + sev]

Notes
- Classical models (SVM/ELM/DBN/AE/BPNN) use PCA features derived from flattened 4-ch scalograms.
- Neural models (CNN, GRU) operate on tensors; they predict both condition and severity.

Run:
  python npy_scalogram_classify_balanced_split_4ch_plus.py
"""

import os, glob, random, json
from pathlib import Path
from typing import List, Tuple, Dict, Optional

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPClassifier, BernoulliRBM
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# -------------------- CONFIG --------------------
SCALOGRAM_ROOT = "scalograms"
ACOUSTIC_DIR   = os.path.join(SCALOGRAM_ROOT, "acoustic")
VIBRATION_DIR  = os.path.join(SCALOGRAM_ROOT, "vibration")

# Channel order for vibration fusion
VIB_CH_ORDER = [
    "x_direction_housing_A",
    "y_direction_housing_A",
    "x_direction_housing_B",
    "y_direction_housing_B",
]

TARGET_PER_CONDITION = 24          # cap per modality per condition (take all if fewer)

# Thinning / cropping
FREQ_SLICE = (None, 128)           # (start, end) rows; None keeps edge
MAX_TIME_COLS_PER_FILE = 20000     # cap columns
COL_STRIDE = 8                     # keep every Nth time column

# Classical/PCA features
PCA_DIM = 128                      # higher to help SVM/ELM/DBN/AE

# Torch training
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS_CNN  = 15
EPOCHS_GRU  = 15
BATCH_SIZE  = 16
BASE_LR     = 3e-4
WEIGHT_DECAY= 1e-2
LABEL_SMOOTH= 0.1
DROPOUT_P   = 0.2
MIXED_PREC  = True
CLIP_NORM   = 1.0

# Data aug (set >0 to enable)
TIME_MASK_PCT = 0.0
FREQ_MASK_PCT = 0.0

# GRU sequence shaping
GRU_TIME_STRIDE = 2    # additional stride across time after COL_STRIDE (to keep T manageable)
GRU_TMAX        = 512  # cap time steps post-stride

# Repro
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)

# -------------------- HELPERS --------------------

def parse_from_meta(meta_path: str):
    with open(meta_path, "r") as fh:
        m = json.load(fh)
    src = m.get("source_csv", "")
    stem = os.path.splitext(os.path.basename(src))[0]
    parts = stem.split("_")
    load = parts[0] if len(parts) > 0 else "UNK"
    cond = parts[1] if len(parts) > 1 else "UNK"
    if cond.lower() == "unbalalnce":
        cond = "Unbalance"
    if cond.lower() == "normal":
        cond = "Normal"
    sev = parts[2] if len(parts) > 2 else "UNK"
    ch  = m.get("channel", None)
    win = int(m.get("window_index", 0))
    return load, cond, sev, ch, win, src

def discover_acoustic_items(acoustic_dir: str) -> List[Tuple[str, str, str]]:
    items = []
    for npy in sorted(glob.glob(os.path.join(acoustic_dir, "*.npy"))):
        meta = npy + ".meta.json"
        if not os.path.exists(meta):
            continue
        try:
            _, cond, sev, _, _, _ = parse_from_meta(meta)
            items.append((npy, cond, sev))
        except Exception:
            pass
    return items

def discover_vibration_groups(vib_dir: str):
    buckets: Dict[Tuple[str,int], Dict[str, Tuple[str,str]]] = {}
    labels: Dict[Tuple[str,int], Tuple[str,str]] = {}
    for npy in sorted(glob.glob(os.path.join(vib_dir, "*.npy"))):
        meta = npy + ".meta.json"
        if not os.path.exists(meta): continue
        try:
            _, cond, sev, ch, win, src = parse_from_meta(meta)
        except Exception:
            continue
        key = (src, win)
        if ch is None:
            continue
        buckets.setdefault(key, {})[ch] = (npy, meta)
        labels[key] = (cond, sev)

    groups: List[List[Tuple[str,str,str]]] = []
    y: List[Tuple[str,str]] = []
    for key, chmap in buckets.items():
        present = [ch for ch in VIB_CH_ORDER if ch in chmap]
        if not present:
            continue
        lst = [(chmap[ch][0], chmap[ch][1], ch) for ch in present]
        groups.append(lst)
        y.append(labels[key])
    return groups, y  # y[i] = (cond, sev) for groups[i]

def freq_time_thin(power: np.ndarray) -> np.ndarray:
    F0, T0 = power.shape
    fs, fe  = FREQ_SLICE
    f_start = 0 if fs is None else max(0, int(fs))
    f_end   = F0 if fe is None else min(F0, int(fe))
    X = power[f_start:f_end, :]
    if MAX_TIME_COLS_PER_FILE is not None:
        T_cap = min(X.shape[1], MAX_TIME_COLS_PER_FILE)
        X = X[:, :T_cap]
    if COL_STRIDE and COL_STRIDE > 1:
        X = X[:, ::int(COL_STRIDE)]
    return X.astype(np.float32, copy=False)

def load_vibration_group_4ch(group: List[Tuple[str,str,str]]) -> np.ndarray:
    mats = []
    order_present = [ch for ch in VIB_CH_ORDER if any(ch == g[2] for g in group)]
    for ch in order_present:
        npy_path = [g[0] for g in group if g[2] == ch][0]
        arr = np.load(npy_path, mmap_mode="r")
        X = freq_time_thin(arr)
        mats.append(X)
    Tm = min(M.shape[1] for M in mats)
    mats = [M[:, :Tm] for M in mats]
    return np.stack(mats, axis=0)  # (C, F, T)

def load_acoustic_as_4ch(npy_path: str) -> np.ndarray:
    arr = np.load(npy_path, mmap_mode="r")
    X = freq_time_thin(arr)  # (F,T)
    out = np.zeros((4, X.shape[0], X.shape[1]), dtype=np.float32)
    out[0] = X
    return out

# -------------------- BALANCED SAMPLING --------------------

def balanced_select(acoustic_items, vib_groups, vib_labels, k=TARGET_PER_CONDITION):
    from collections import defaultdict
    rng = np.random.default_rng(SEED)
    selected = []

    by_cond_a = defaultdict(list)
    for npy_path, cond, sev in acoustic_items:
        by_cond_a[cond].append((npy_path, cond, sev))
    for cond, lst in by_cond_a.items():
        lst = lst.copy(); rng.shuffle(lst)
        take = lst[:k] if len(lst) >= k else lst
        for npy_path, c, s in take:
            selected.append(("acoustic", npy_path, c, s))

    by_cond_v = defaultdict(list)
    for g, (cond, sev) in zip(vib_groups, vib_labels):
        by_cond_v[cond].append((g, cond, sev))
    for cond, lst in by_cond_v.items():
        lst = lst.copy(); rng.shuffle(lst)
        take = lst[:k] if len(lst) >= k else lst
        for g, c, s in take:
            selected.append(("vibration", g, c, s))

    rng.shuffle(selected)
    return selected

# -------------------- DATASETS --------------------
class NpyDataset4Ch(Dataset):
    def __init__(self, items, cond2id: Dict[str,int], sev2id: Dict[str,int]):
        self.items = items
        self.cond2id = cond2id
        self.sev2id  = sev2id

    def __len__(self): return len(self.items)

    def __getitem__(self, i):
        src_type, payload, cond, sev = self.items[i]
        if src_type == "acoustic":
            X = load_acoustic_as_4ch(payload)  # (4,F,T)
        else:
            X = load_vibration_group_4ch(payload)
            if X.shape[0] < 4:
                C, F, T = X.shape
                tmp = np.zeros((4, F, T), dtype=np.float32)
                tmp[:C] = X
                X = tmp
        yc = self.cond2id[cond]
        ys = self.sev2id[sev]
        return torch.from_numpy(X), torch.tensor(yc, dtype=torch.long), torch.tensor(ys, dtype=torch.long)

# -------------------- CLASSICAL FEATURE STACK --------------------
def build_flat_features(items_subset):
    """
    Returns matrix (N, 4*F*T) after log1p.
    """
    mats = []
    for src_type, payload, _, _ in items_subset:
        if src_type == "acoustic":
            X = load_acoustic_as_4ch(payload)
        else:
            X = load_vibration_group_4ch(payload)
            if X.shape[0] < 4:
                C, F, T = X.shape
                tmp = np.zeros((4, F, T), dtype=np.float32)
                tmp[:C] = X
                X = tmp
        X = np.log1p(X).astype(np.float32)
        mats.append(X.reshape(-1))
    return np.stack(mats)

# -------------------- MODELS --------------------
class CNN2D(nn.Module):
    def __init__(self, n_cond: int, n_sev: int, in_ch: int = 4):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(in_ch, 32, 5, padding=2, stride=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d((2,4)), nn.Dropout(DROPOUT_P),

            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d((2,2)), nn.Dropout(DROPOUT_P),

            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 8)),
        )
        self.flat = nn.Flatten()
        self.fc_cond = nn.Linear(128*4*8, n_cond)
        self.fc_sev  = nn.Linear(128*4*8, n_sev)

    def forward(self, x):
        z = self.backbone(x)
        z = self.flat(z)
        return self.fc_cond(z), self.fc_sev(z)

class GRUClassifier(nn.Module):
    """
    Treat time as sequence; at each step feature = concat over channels and freq.
    Input expected: x (B, 4, F, T) -> reshape to (B, T', 4*F), with extra stride and cap.
    """
    def __init__(self, n_cond: int, n_sev: int, feat_dim: int, hidden: int = 128, layers: int = 2, bidir: bool = True):
        super().__init__()
        self.gru = nn.GRU(
            input_size=feat_dim,
            hidden_size=hidden,
            num_layers=layers,
            batch_first=True,
            dropout=DROPOUT_P if layers > 1 else 0.0,
            bidirectional=bidir
        )
        mult = 2 if bidir else 1
        self.drop = nn.Dropout(DROPOUT_P)
        self.fc_cond = nn.Linear(hidden*mult, n_cond)
        self.fc_sev  = nn.Linear(hidden*mult, n_sev)

    def forward(self, x):
        # x: (B,4,F,T)
        # normalize per-channel (log1p + zscore) then prepare sequence
        x = torch.log1p(torch.clamp(x, min=0))
        mu = x.mean(dim=(2,3), keepdim=True)
        sd = x.std(dim=(2,3), keepdim=True) + 1e-6
        x = (x - mu) / sd
        B, C, F, T = x.shape
        # additional time stride
        if GRU_TIME_STRIDE > 1:
            x = x[..., ::GRU_TIME_STRIDE]  # (B,4,F,T')
        if x.shape[-1] > GRU_TMAX:
            x = x[..., :GRU_TMAX]
        B, C, F, T2 = x.shape
        x = x.permute(0, 3, 1, 2).contiguous().view(B, T2, C*F)  # (B, T2, 4*F)

        out, _ = self.gru(x)  # (B, T2, H*mult)
        h = out.mean(dim=1)   # global mean pool across time
        h = self.drop(h)
        return self.fc_cond(h), self.fc_sev(h)

# -------------------- TRAIN/EVAL (TORCH) --------------------
def spec_mask(x: torch.Tensor) -> torch.Tensor:
    # x: (B,C,F,T) for CNN; (B,T,feat) already handled inside GRU; we use only for CNN pipeline.
    if TIME_MASK_PCT <= 0 and FREQ_MASK_PCT <= 0:
        return x
    B, C, F, T = x.shape
    if FREQ_MASK_PCT > 0:
        w = max(1, int(F * FREQ_MASK_PCT))
        for b in range(B):
            f0 = np.random.randint(0, max(1, F - w + 1))
            x[b, :, f0:f0+w, :] = 0
    if TIME_MASK_PCT > 0:
        w = max(1, int(T * TIME_MASK_PCT))
        for b in range(B):
            t0 = np.random.randint(0, max(1, T - w + 1))
            x[b, :, :, t0:t0+w] = 0
    return x

def normalize_batch_per_channel(x: torch.Tensor) -> torch.Tensor:
    x = torch.log1p(torch.clamp(x, min=0))
    mu = x.mean(dim=(2,3), keepdim=True)
    sd = x.std(dim=(2,3), keepdim=True) + 1e-6
    return (x - mu) / sd

def train_multitask(model, dl_tr, dl_va, epochs=15):
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=max(1, epochs//2))
    crit = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE=="cuda" and MIXED_PREC))

    best = None; best_val = 1e9; patience=4; wait=0
    for ep in range(1, epochs+1):
        model.train(); tr_sum=0.0; n=0
        for xb, yc, ys in dl_tr:
            xb, yc, ys = xb.to(DEVICE), yc.to(DEVICE), ys.to(DEVICE)
            # For CNN we'll pass raw then normalize here; GRU normalizes inside forward.
            if isinstance(model, CNN2D):
                xb = normalize_batch_per_channel(xb)
                xb = spec_mask(xb)

            opt.zero_grad(set_to_none=True)
            if scaler.is_enabled():
                with torch.cuda.amp.autocast():
                    pc, ps = model(xb)
                    loss = crit(pc, yc) + crit(ps, ys)
                scaler.scale(loss).backward()
                nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
                scaler.step(opt); scaler.update()
            else:
                pc, ps = model(xb)
                loss = crit(pc, yc) + crit(ps, ys)
                loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
                opt.step()

            n += xb.size(0)
            sched.step(ep - 1 + n / max(1, len(dl_tr.dataset)))
            tr_sum += float(loss.item()) * xb.size(0)
        tr = tr_sum / max(1, n)

        # val
        model.eval(); va_sum=0.0; nva=0
        with torch.no_grad():
            for xb, yc, ys in dl_va:
                xb, yc, ys = xb.to(DEVICE), yc.to(DEVICE), ys.to(DEVICE)
                if isinstance(model, CNN2D):
                    xb = normalize_batch_per_channel(xb)
                pc, ps = model(xb)
                loss = nn.functional.cross_entropy(pc, yc) + nn.functional.cross_entropy(ps, ys)
                va_sum += float(loss.item()) * xb.size(0); nva += xb.size(0)
        va = va_sum / max(1,nva)
        print(f"[{model.__class__.__name__}] epoch {ep:02d}  train={tr:.4f}  val={va:.4f}")

        if va < best_val - 1e-6:
            best_val = va; best = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; wait=0
        else:
            wait += 1
            if wait >= patience:
                print("  early stop"); break
    if best is not None:
        model.load_state_dict(best)
    return model

def eval_multitask(model, dl):
    model.eval(); yc_true, yc_pred, ys_true, ys_pred = [], [], [], []
    with torch.no_grad():
        for xb, yc, ys in dl:
            xb = xb.to(DEVICE)
            if isinstance(model, CNN2D):
                xb = normalize_batch_per_channel(xb)
            pc, ps = model(xb)
            yc_pred.extend(pc.argmax(1).cpu().numpy().tolist())
            ys_pred.extend(ps.argmax(1).cpu().numpy().tolist())
            yc_true.extend(yc.numpy().tolist())
            ys_true.extend(ys.numpy().tolist())
    return np.array(yc_true), np.array(yc_pred), np.array(ys_true), np.array(ys_pred)

def report(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1m = f1_score(y_true, y_pred, average="macro")
    print(f"{name:>12s} | Acc={acc:.3f}  F1(macro)={f1m:.3f}")
    return acc, f1m

# -------------------- ELM (simple numpy implementation) --------------------
class ELMClassifier:
    """
    Single hidden layer ELM with random weights and ridge regression output.
    Use on PCA features (low-dim).
    """
    def __init__(self, n_hidden=512, alpha=1e-2, activation="relu", seed=SEED):
        self.n_hidden = n_hidden
        self.alpha = alpha
        self.activation = activation
        self.rng = np.random.default_rng(seed)
        self.W = None
        self.b = None
        self.beta = None  # output weights (hidden->classes)

    def _act(self, Z):
        if self.activation == "relu":
            return np.maximum(0, Z)
        elif self.activation == "tanh":
            return np.tanh(Z)
        else:  # linear
            return Z

    def fit(self, X, y):
        n, d = X.shape
        classes = np.unique(y)
        n_classes = classes.size
        self.classes_ = classes

        # one-hot
        Y = np.zeros((n, n_classes), dtype=np.float64)
        for i, c in enumerate(classes):
            Y[y == c, i] = 1.0

        # random hidden layer
        self.W = self.rng.standard_normal((d, self.n_hidden)) / np.sqrt(d)
        self.b = self.rng.standard_normal((self.n_hidden,))
        H = self._act(X @ self.W + self.b)

        # ridge regression: beta = (H^T H + alpha I)^-1 H^T Y
        A = H.T @ H + self.alpha * np.eye(self.n_hidden)
        B = H.T @ Y
        self.beta = np.linalg.solve(A, B)  # (n_hidden, n_classes)

    def predict(self, X):
        H = self._act(X @ self.W + self.b)
        logits = H @ self.beta
        return self.classes_[np.argmax(logits, axis=1)]

# -------------------- Autoencoder on PCA features --------------------
class AE(nn.Module):
    def __init__(self, in_dim=128, bottleneck=32):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(in_dim, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, bottleneck),
        )
        self.dec = nn.Sequential(
            nn.Linear(bottleneck, 64), nn.ReLU(),
            nn.Linear(64, 128), nn.ReLU(),
            nn.Linear(128, in_dim),
        )
    def forward(self, x):
        z = self.enc(x)
        r = self.dec(z)
        return r, z

def train_autoencoder(Xtr_p, Xva_p, epochs=40, lr=1e-3, bottleneck=32):
    device = DEVICE
    Xtr = torch.from_numpy(Xtr_p.astype(np.float32)).to(device)
    Xva = torch.from_numpy(Xva_p.astype(np.float32)).to(device)
    model = AE(in_dim=Xtr.shape[1], bottleneck=bottleneck).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    best = None; best_val = 1e9; wait=0; patience=6
    for ep in range(1, epochs+1):
        model.train()
        opt.zero_grad()
        r, z = model(Xtr)
        loss = nn.functional.mse_loss(r, Xtr)
        loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            rv, zv = model(Xva)
            vloss = nn.functional.mse_loss(rv, Xva).item()
        print(f"[AE] epoch {ep:02d} train={loss.item():.5f} val={vloss:.5f}")
        if vloss < best_val - 1e-6:
            best_val = vloss; best = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; wait=0
        else:
            wait += 1
            if wait >= patience:
                print("  AE early stop"); break
    if best: model.load_state_dict(best)
    model.eval()
    with torch.no_grad():
        _, Ztr = model(Xtr); _, Zva = model(Xva)
    return model, Ztr.cpu().numpy(), Zva.cpu().numpy()

# -------------------- PIPELINE --------------------

def main():
    # Discover items
    acoustic_items = discover_acoustic_items(ACOUSTIC_DIR)
    vib_groups, vib_labels = discover_vibration_groups(VIBRATION_DIR)

    if not acoustic_items and not vib_groups:
        print("[ERROR] No .npy scalograms found.")
        return

    # Vocabulary
    conditions = sorted(set([c for _, c, _ in acoustic_items] + [c for (c, _) in vib_labels]))
    severities = sorted(set([s for _, _, s in acoustic_items] + [s for (_, s) in vib_labels]))
    cond2id = {c:i for i,c in enumerate(conditions)}
    sev2id  = {s:i for i,s in enumerate(severities)}

    # Balanced selection per modality & condition
    selected = balanced_select(acoustic_items, vib_groups, vib_labels, k=TARGET_PER_CONDITION)
    if not selected:
        selected = [("acoustic", npy, c, s) for (npy, c, s) in acoustic_items] + \
                   [("vibration", g, c, s) for g, (c, s) in zip(vib_groups, vib_labels)]

    # Stratified split by condition
    y_cond = np.array([cond2id[c] for (_, _, c, _) in selected], dtype=np.int64)
    idx = np.arange(len(selected))

    def safe_split(indices, strat_labels, test_size):
        try:
            return train_test_split(indices, test_size=test_size, random_state=SEED, stratify=strat_labels)
        except Exception:
            return train_test_split(indices, test_size=test_size, random_state=SEED)

    tr_idx, te_idx = safe_split(idx, y_cond, test_size=0.15)
    tr_idx, va_idx = safe_split(tr_idx, y_cond[tr_idx], test_size=0.1765)  # ~70/15/15

    tr_items = [selected[i] for i in tr_idx]
    va_items = [selected[i] for i in va_idx]
    te_items = [selected[i] for i in te_idx]

    # Datasets & loaders for neural models
    ds_tr = NpyDataset4Ch(tr_items, cond2id, sev2id)
    ds_va = NpyDataset4Ch(va_items, cond2id, sev2id)
    ds_te = NpyDataset4Ch(te_items, cond2id, sev2id)
    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    dl_va = DataLoader(ds_va, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    dl_te = DataLoader(ds_te, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    # ===== Classical feature stack (PCA) =====
    print("\n[Build PCA features for classical models]")
    Xtr_f = build_flat_features(tr_items)
    Xva_f = build_flat_features(va_items)
    Xte_f = build_flat_features(te_items)

    # Standardize -> PCA
    std = StandardScaler(with_mean=True, with_std=True)
    Xtr_s = std.fit_transform(Xtr_f); Xva_s = std.transform(Xva_f); Xte_s = std.transform(Xte_f)
    pca = PCA(n_components=min(PCA_DIM, Xtr_s.shape[1]), random_state=SEED)
    Xtr_p = pca.fit_transform(Xtr_s); Xva_p = pca.transform(Xva_s); Xte_p = pca.transform(Xte_s)

    yctr = np.array([cond2id[c] for (_,_,c,_) in tr_items], dtype=np.int64)
    ycva = np.array([cond2id[c] for (_,_,c,_) in va_items], dtype=np.int64)
    ycte = np.array([cond2id[c] for (_,_,c,_) in te_items], dtype=np.int64)

    # ===== BPNN (MLP) baseline =====
    mlp = MLPClassifier(hidden_layer_sizes=(256,128), activation="relu", max_iter=600, random_state=SEED)
    mlp.fit(Xtr_p, yctr)
    print("\n=== Classical models on PCA features ===")
    report("BPNN(cond)", ycte, mlp.predict(Xte_p))

    # ===== SVM (RBF) =====
    svm = SVC(C=4.0, gamma="scale", kernel="rbf", class_weight="balanced", random_state=SEED)
    svm.fit(Xtr_p, yctr)
    report("SVM (cond)", ycte, svm.predict(Xte_p))

    # ===== ELM =====
    elm = ELMClassifier(n_hidden=512, alpha=1e-2, activation="relu", seed=SEED)
    elm.fit(Xtr_p, yctr)
    report("ELM (cond)", ycte, elm.predict(Xte_p))

    # ===== DBN (RBM stack + LR) =====
    # Scale to [0,1] for BernoulliRBM
    mm = MinMaxScaler()
    Xtr_b = mm.fit_transform(Xtr_p); Xva_b = mm.transform(Xva_p); Xte_b = mm.transform(Xte_p)
    rbm1 = BernoulliRBM(n_components=256, learning_rate=0.05, n_iter=20, batch_size=32, random_state=SEED, verbose=False)
    rbm2 = BernoulliRBM(n_components=128, learning_rate=0.05, n_iter=20, batch_size=32, random_state=SEED, verbose=False)
    lr   = LogisticRegression(max_iter=200, class_weight="balanced", random_state=SEED)
    dbn = Pipeline(steps=[("rbm1", rbm1), ("rbm2", rbm2), ("lr", lr)])
    dbn.fit(Xtr_b, yctr)
    report("DBN (cond)", ycte, dbn.predict(Xte_b))

    # ===== AE + Logistic Regression =====
    ae, Ztr, Zva = train_autoencoder(Xtr_p, Xva_p, epochs=50, lr=1e-3, bottleneck=32)
    Zte = ae.enc(torch.from_numpy(Xte_p.astype(np.float32)).to(DEVICE)).detach().cpu().numpy()
    ae_lr = LogisticRegression(max_iter=200, class_weight="balanced", random_state=SEED)
    ae_lr.fit(Ztr, yctr)
    report("AE+LR(cond)", ycte, ae_lr.predict(Zte))

    # ===== CNN (4-ch, multitask) =====
    print("\n=== Neural models (multitask) ===")
    cnn = CNN2D(n_cond=len(conditions), n_sev=len(severities), in_ch=4)
    cnn = train_multitask(cnn, dl_tr, dl_va, epochs=EPOCHS_CNN)
    yc_true, yc_pred, ys_true, ys_pred = eval_multitask(cnn, dl_te)
    report("CNN(cond)", yc_true, yc_pred)
    report("CNN(sev)",  ys_true, ys_pred)

    # ===== GRU (sequence over time, multitask) =====
    # For GRU, we reuse the same dataset; GRU handles normalization/reshape internally.
    # feature dim = 4*F where F is variable per batch; to make it stable, we collate to max F across batch by truncation.
    # Simpler: enforce constant F by pre-thinning; our loader already aligns per sample, but across batch F may differ.
    # We implement a custom collate that zero-pads/truncates F to min F in batch.

    def collate_constant_F(batch):
        # batch: list of (X(4,F,T), yc, ys)
        Fs = [x[0].shape[1] for x in batch]
        Fm = min(Fs)  # truncate to smallest freq bins in batch
        Xs = []
        for x, yc, ys in batch:
            Xs.append(x[:, :Fm, :])  # (4,Fm,T)
        X = torch.stack(Xs, dim=0)  # (B,4,Fm,T)
        ycs = torch.stack([y for _, y, _ in batch], dim=0)
        yss = torch.stack([y for _, _, y in batch], dim=0)
        return X, ycs, yss

    dl_tr_gru = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, collate_fn=collate_constant_F)
    dl_va_gru = DataLoader(ds_va, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_constant_F)
    dl_te_gru = DataLoader(ds_te, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_constant_F)

    # peek F to build GRU
    X0, _, _ = next(iter(dl_tr_gru))
    feat_dim = X0.shape[1] * X0.shape[2]  # 4 * F
    gru = GRUClassifier(n_cond=len(conditions), n_sev=len(severities), feat_dim=feat_dim, hidden=128, layers=2, bidir=True).to(DEVICE)
    gru = train_multitask(gru, dl_tr_gru, dl_va_gru, epochs=EPOCHS_GRU)
    yc_true_g, yc_pred_g, ys_true_g, ys_pred_g = eval_multitask(gru, dl_te_gru)
    report("GRU(cond)", yc_true_g, yc_pred_g)
    report("GRU(sev)",  ys_true_g, ys_pred_g)

    print("\nDone.")

if __name__ == "__main__":
    main()



[Build PCA features for classical models]

=== Classical models on PCA features ===
  BPNN(cond) | Acc=0.828  F1(macro)=0.812
  SVM (cond) | Acc=0.862  F1(macro)=0.849
  ELM (cond) | Acc=0.862  F1(macro)=0.817
  DBN (cond) | Acc=0.310  F1(macro)=0.259
[AE] epoch 01 train=6400.85596 val=5034.30127
[AE] epoch 02 train=6391.53809 val=5025.46680
[AE] epoch 03 train=6382.87500 val=5015.41064
[AE] epoch 04 train=6373.20508 val=5003.70703
[AE] epoch 05 train=6361.61914 val=4989.70850
[AE] epoch 06 train=6347.49414 val=4972.81396
[AE] epoch 07 train=6330.40332 val=4952.20459
[AE] epoch 08 train=6309.69971 val=4927.17627
[AE] epoch 09 train=6284.61865 val=4896.85645
[AE] epoch 10 train=6254.32715 val=4860.49902
[AE] epoch 11 train=6217.93213 val=4816.45410
[AE] epoch 12 train=6174.19971 val=4763.55273
[AE] epoch 13 train=6121.70459 val=4700.24561
[AE] epoch 14 train=6058.95410 val=4624.72998
[AE] epoch 15 train=5984.23242 val=4535.26367
[AE] epoch 16 train=5895.51855 val=4429.35059
[AE] epoch 

C:\Users\gsahi\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\gsahi\AppData\Local\Temp\ipykernel_25448\3213573968.py:350: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE=="cuda" and MIXED_PREC))


 AE+LR(cond) | Acc=0.690  F1(macro)=0.567

=== Neural models (multitask) ===


C:\Users\gsahi\AppData\Local\Temp\ipykernel_25448\3213573968.py:364: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


[CNN2D] epoch 01  train=3.5534  val=3.3506
[CNN2D] epoch 02  train=3.1171  val=2.9554
[CNN2D] epoch 03  train=2.8576  val=2.6896
[CNN2D] epoch 04  train=2.6830  val=2.4821
[CNN2D] epoch 05  train=2.5434  val=2.3675
[CNN2D] epoch 06  train=2.5427  val=2.3082
[CNN2D] epoch 07  train=2.4790  val=2.3179
[CNN2D] epoch 08  train=2.4245  val=1.9611
[CNN2D] epoch 09  train=2.2000  val=1.7295
[CNN2D] epoch 10  train=2.0590  val=1.6374
[CNN2D] epoch 11  train=1.9257  val=1.5643
[CNN2D] epoch 12  train=1.8643  val=1.4999
[CNN2D] epoch 13  train=1.8232  val=1.4712
[CNN2D] epoch 14  train=1.8154  val=1.4751
[CNN2D] epoch 15  train=1.8218  val=1.4193
   CNN(cond) | Acc=0.862  F1(macro)=0.756
    CNN(sev) | Acc=0.828  F1(macro)=0.534


C:\Users\gsahi\AppData\Local\Temp\ipykernel_25448\3213573968.py:350: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE=="cuda" and MIXED_PREC))
C:\Users\gsahi\AppData\Local\Temp\ipykernel_25448\3213573968.py:364: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


[GRUClassifier] epoch 01  train=3.4576  val=3.1348
[GRUClassifier] epoch 02  train=3.0553  val=2.7981
[GRUClassifier] epoch 03  train=2.8060  val=2.5785
[GRUClassifier] epoch 04  train=2.6287  val=2.3954
[GRUClassifier] epoch 05  train=2.5014  val=2.2894
[GRUClassifier] epoch 06  train=2.4445  val=2.2501
[GRUClassifier] epoch 07  train=2.4293  val=2.2441
[GRUClassifier] epoch 08  train=2.3486  val=1.9439
[GRUClassifier] epoch 09  train=2.0905  val=1.7012
[GRUClassifier] epoch 10  train=1.9425  val=1.5214
[GRUClassifier] epoch 11  train=1.8267  val=1.4200
[GRUClassifier] epoch 12  train=1.7686  val=1.3703
[GRUClassifier] epoch 13  train=1.7382  val=1.3474
[GRUClassifier] epoch 14  train=1.7206  val=1.3441
[GRUClassifier] epoch 15  train=1.6770  val=1.1934
   GRU(cond) | Acc=0.897  F1(macro)=0.845
    GRU(sev) | Acc=0.862  F1(macro)=0.683

Done.
